# PyNRPF v0.1.0 - Publication Tables CSV Export

This notebook creates four CSV tables for the paper:
1. Dataset train-test split summary (computed from dataset)
2. Model performance summary (loaded from metrics JSON)
3. Implementation MW summary (2023, manual/anonymized examples)
4. Implementation timestamp summary (2023, manual/anonymized examples)

In [1]:
# Environment and imports
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.io import load_yaml, req, ensure_dir, load_parquet

print('Python:', sys.version)
print('CWD:   ', Path.cwd())
print('REPO:  ', REPO_ROOT)

Python: 3.12.3 (tags/v3.12.3:f6650f9, Apr  9 2024, 14:05:25) [MSC v.1938 64 bit (AMD64)]
CWD:    C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\notebooks
REPO:   C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper


In [2]:
# Config and paths
CFG_PATH = REPO_ROOT / 'config' / 'run.yaml'
cfg = load_yaml(CFG_PATH)

RUN_TAG = str(req(cfg, 'run.run_tag'))

DATASET_PATH = (REPO_ROOT / str(req(cfg, 'paths.dataset_parquet'))).resolve()
OUTPUT_DIR = (REPO_ROOT / str(req(cfg, 'paths.output_dir'))).resolve()
TABLE_DIR = OUTPUT_DIR / 'publication_tables'
ensure_dir(TABLE_DIR)

METRICS_JSON = OUTPUT_DIR / f'metrics__{RUN_TAG}.json'

COL_SITE = str(req(cfg, 'data.columns.site'))
COL_TS = str(req(cfg, 'data.columns.ts'))
COL_GT = str(req(cfg, 'data.columns.gt'))

TRAIN_START = pd.Timestamp(str(req(cfg, 'split.train_start'))).date()
TRAIN_END = pd.Timestamp(str(req(cfg, 'split.train_end'))).date()
TEST_START = pd.Timestamp(str(req(cfg, 'split.test_start'))).date()
TEST_END = pd.Timestamp(str(req(cfg, 'split.test_end'))).date()

print('Config loaded from:', CFG_PATH)
print('Run tag:', RUN_TAG)
print('Dataset:', DATASET_PATH)
print('Metrics:', METRICS_JSON)
print('Output table dir:', TABLE_DIR)
print(f'Split: train {TRAIN_START}..{TRAIN_END} | test {TEST_START}..{TEST_END}')

Config loaded from: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\config\run.yaml
Run tag: local_dev
Dataset: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\dataset\raw\rpf_dataset.parquet
Metrics: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\metrics__local_dev.json
Output table dir: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\publication_tables
Split: train 2021-11-01..2023-09-30 | test 2023-10-01..2024-09-30


In [3]:
# Load dataset and split
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

if not METRICS_JSON.exists():
    raise FileNotFoundError(f'Metrics JSON not found: {METRICS_JSON}')

df = load_parquet(DATASET_PATH)
df[COL_TS] = pd.to_datetime(df[COL_TS], errors='raise')
df['date'] = df[COL_TS].dt.date

train_mask = (df['date'] >= TRAIN_START) & (df['date'] <= TRAIN_END)
test_mask = (df['date'] >= TEST_START) & (df['date'] <= TEST_END)

df_train = df.loc[train_mask].copy()
df_test = df.loc[test_mask].copy()
n_other = int((~train_mask & ~test_mask).sum())

print(f'Train rows: {len(df_train):,}')
print(f'Test rows:  {len(df_test):,}')
print(f'Other rows: {n_other:,}')

Loaded 1,011,264 rows x 5 cols from rpf_dataset.parquet


Train rows: 666,432
Test rows:  344,832
Other rows: 0


In [4]:
# Table 1 - Dataset train-test split summary
def build_split_rows(df_split: pd.DataFrame, split_name: str) -> list[dict]:
    interval_total = int(len(df_split))
    interval_need = int((df_split[COL_GT] < 0).sum())
    interval_pct = round((interval_need / interval_total * 100.0) if interval_total else 0.0, 3)

    day_df = (
        df_split.groupby([COL_SITE, 'date'])
        .agg(need_correction=(COL_GT, lambda s: bool((s < 0).any())))
        .reset_index()
    )
    day_total = int(len(day_df))
    day_need = int(day_df['need_correction'].sum())
    day_pct = round((day_need / day_total * 100.0) if day_total else 0.0, 3)

    return [
        {
            'split': split_name,
            'level': 'day',
            'n_total': day_total,
            'n_need_correction': day_need,
            'pct_need_correction': day_pct,
        },
        {
            'split': split_name,
            'level': 'interval',
            'n_total': interval_total,
            'n_need_correction': interval_need,
            'pct_need_correction': interval_pct,
        },
    ]

table1_rows = []
table1_rows.extend(build_split_rows(df_train, 'train'))
table1_rows.extend(build_split_rows(df_test, 'test'))

table1 = pd.DataFrame(table1_rows)

level_order = {'day': 0, 'interval': 1}
split_order = {'train': 0, 'test': 1}
table1 = table1.sort_values(
    by=['level', 'split'],
    key=lambda s: s.map(level_order if s.name == 'level' else split_order),
).reset_index(drop=True)

table1 = table1[['level', 'split', 'n_total', 'n_need_correction', 'pct_need_correction']]
table1

,level,split,n_total,n_need_correction,pct_need_correction
0,day,train,6986,2095,29.989
1,day,test,3657,1328,36.314
2,interval,train,666432,27678,4.153
3,interval,test,344832,20302,5.888


In [5]:
# Table 2 - Model performance summary (from metrics JSON, unpivoted split rows)
with METRICS_JSON.open('r', encoding='utf-8') as f:
    metrics = json.load(f)

method_name_map = {
    'm7_threshold': 'm7_dtr',
    'm8_xgb': 'm8_xgb',
}

def metric_rows(method_key: str, model_level: str, interval_scope: str, result_key: str) -> list[dict]:
    section = metrics[method_key][result_key]
    rows = []
    for split in ['train', 'test']:
        part = section[split]
        rows.append({
            'method': method_name_map.get(method_key, method_key),
            'model_level': model_level,
            'interval_scope': interval_scope,
            'split': split,
            'P': round(float(part['precision']), 3),
            'R': round(float(part['recall']), 3),
            'F1': round(float(part['f1']), 3),
        })
    return rows

base_specs = [
    ('m7_threshold', 'day', 'na', 'day'),
    ('m7_threshold', 'interval', 'all_days', 'interval_all_days'),
    ('m7_threshold', 'interval', 'tp_days_only', 'interval_tp_days_only'),
    ('m8_xgb', 'day', 'na', 'day'),
    ('m8_xgb', 'interval', 'all_days', 'interval_all_days'),
    ('m8_xgb', 'interval', 'tp_days_only', 'interval_tp_days_only'),
]

table2_rows = []
for method_key, model_level, interval_scope, result_key in base_specs:
    table2_rows.extend(metric_rows(method_key, model_level, interval_scope, result_key))

table2 = pd.DataFrame(table2_rows)

split_order = ['train', 'test']
model_level_order = ['day', 'interval']
interval_scope_order = ['na', 'all_days', 'tp_days_only']
method_order = ['m7_dtr', 'm8_xgb']

for col, order in [
    ('split', split_order),
    ('model_level', model_level_order),
    ('interval_scope', interval_scope_order),
    ('method', method_order),
]:
    table2[col] = pd.Categorical(table2[col], categories=order, ordered=True)

table2 = table2.sort_values(
    by=['split', 'model_level', 'interval_scope', 'method']
).reset_index(drop=True)

table2 = table2[[
    'split', 'model_level', 'interval_scope', 'method',
    'P', 'R', 'F1',
]]
table2


,split,model_level,interval_scope,method,P,R,F1
0,train,day,na,m7_dtr,0.889,0.980,0.932
1,train,day,na,m8_xgb,1.000,1.000,1.000
2,train,interval,all_days,m7_dtr,0.807,0.770,0.788
3,train,interval,all_days,m8_xgb,0.967,0.838,0.898
4,train,interval,tp_days_only,m7_dtr,0.840,0.770,0.804
5,train,interval,tp_days_only,m8_xgb,0.967,0.838,0.898
6,test,day,na,m7_dtr,0.912,0.963,0.937
7,test,day,na,m8_xgb,0.954,0.958,0.956
8,test,interval,all_days,m7_dtr,0.844,0.796,0.819
9,test,interval,all_days,m8_xgb,0.922,0.796,0.854


In [6]:
# Table 3 / Table 4 - Implementation result (2023, manual/anonymized examples)
table_base = pd.DataFrame([
    {
        'site_name': 'network',
        'min_demand_MW_before': 1481.0,
        'min_demand_MW_after': 1447.0,
        'ts_min_before': '',
        'ts_min_after': '',
    },
    {
        'site_name': 'Site X',
        'min_demand_MW_before': 0.00,
        'min_demand_MW_after': -8.21,
        'ts_min_before': '25/10/2023 13:00',
        'ts_min_after': '21/11/2023 14:15',
    },
    {
        'site_name': 'Site Y',
        'min_demand_MW_before': 0.00,
        'min_demand_MW_after': -6.41,
        'ts_min_before': '17/05/2023 8:15',
        'ts_min_after': '5/10/2023 12:30',
    },
    {
        'site_name': 'Site Z',
        'min_demand_MW_before': 0.00,
        'min_demand_MW_after': -5.24,
        'ts_min_before': '30/11/2023 9:30',
        'ts_min_after': '6/03/2023 13:45',
    },
])

table_base['min_demand_MW_correction'] = (
    table_base['min_demand_MW_after'] - table_base['min_demand_MW_before']
).round(2)

# Table 3: MW value difference (no year column)
table3 = table_base[[
    'site_name',
    'min_demand_MW_before', 'min_demand_MW_after', 'min_demand_MW_correction',
]].copy()

# Table 4: Timestamp difference (no year column; exclude network row)
table4 = table_base.loc[
    table_base['site_name'] != 'network',
    ['site_name', 'ts_min_before', 'ts_min_after'],
].copy()

table3

,site_name,min_demand_MW_before,min_demand_MW_after,min_demand_MW_correction
0,network,1481.0,1447.00,-34.00
1,Site X,0.0,-8.21,-8.21
2,Site Y,0.0,-6.41,-6.41
3,Site Z,0.0,-5.24,-5.24


In [7]:
# Export CSV files
TABLE1_PATH = TABLE_DIR / 'table1_dataset_split_summary.csv'
TABLE2_PATH = TABLE_DIR / 'table2_model_performance_summary.csv'
TABLE3_PATH = TABLE_DIR / 'table3_implementation_mw_difference.csv'
TABLE4_PATH = TABLE_DIR / 'table4_implementation_ts_difference.csv'


def _title_header(name: str) -> str:
    parts = name.split('_')
    return ' '.join([(p[:1].upper() + p[1:]) if p else p for p in parts])


def _with_title_headers(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns={c: _title_header(c) for c in df.columns})


def _safe_write_csv(df: pd.DataFrame, path: Path) -> Path:
    try:
        df.to_csv(path, index=False)
        return path
    except PermissionError:
        fallback = path.with_name(path.stem + '_updated.csv')
        df.to_csv(fallback, index=False)
        print(f'Permission denied for {path}; wrote to {fallback} instead')
        return fallback


# Apply publication-style headers: capitalized words with spaces (no underscores).
table1_export = _with_title_headers(table1)
table2_export = _with_title_headers(table2)
table3_export = _with_title_headers(table3)
table4_export = _with_title_headers(table4)


# Remove previous table CSV variants; keep only current table1-4 outputs.
keep_names = {TABLE1_PATH.name, TABLE2_PATH.name, TABLE3_PATH.name, TABLE4_PATH.name}
for old_csv in TABLE_DIR.glob('table*.csv'):
    if old_csv.name not in keep_names:
        old_csv.unlink(missing_ok=True)

archive_dir = TABLE_DIR / 'archived'
if archive_dir.exists():
    for old_csv in archive_dir.glob('table*.csv'):
        old_csv.unlink(missing_ok=True)


TABLE1_WRITTEN = _safe_write_csv(table1_export, TABLE1_PATH)
TABLE2_WRITTEN = _safe_write_csv(table2_export, TABLE2_PATH)
TABLE3_WRITTEN = _safe_write_csv(table3_export, TABLE3_PATH)
TABLE4_WRITTEN = _safe_write_csv(table4_export, TABLE4_PATH)

# Ensure no *_updated file remains for table2 (canonical filename only).
if TABLE2_WRITTEN.name.endswith('_updated.csv'):
    canonical_t2 = TABLE2_PATH
    TABLE2_WRITTEN.replace(canonical_t2)
    TABLE2_WRITTEN = canonical_t2

for d in [TABLE_DIR, archive_dir]:
    if d and d.exists():
        upd = d / 'table2_model_performance_summary_updated.csv'
        if upd.exists():
            upd.unlink(missing_ok=True)

print(f'Wrote: {TABLE1_WRITTEN}')
print(f'Wrote: {TABLE2_WRITTEN}')
print(f'Wrote: {TABLE3_WRITTEN}')
print(f'Wrote: {TABLE4_WRITTEN}')

Wrote: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\publication_tables\table1_dataset_split_summary.csv
Wrote: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\publication_tables\table2_model_performance_summary.csv
Wrote: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\publication_tables\table3_implementation_mw_difference.csv
Wrote: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\publication_tables\table4_implementation_ts_difference.csv


In [8]:
# Validation checks
assert TABLE1_WRITTEN.exists(), 'Missing table1 CSV'
assert TABLE2_WRITTEN.exists(), 'Missing table2 CSV'
assert TABLE3_WRITTEN.exists(), 'Missing table3 CSV'
assert TABLE4_WRITTEN.exists(), 'Missing table4 CSV'

assert len(table1_export) == 4, f'Table 1 row count expected 4, got {len(table1_export)}'
assert len(table2_export) == 12, f'Table 2 row count expected 12, got {len(table2_export)}'
assert len(table3_export) == 4, f'Table 3 row count expected 4, got {len(table3_export)}'
assert len(table4_export) == 3, f'Table 4 row count expected 3, got {len(table4_export)}'

req_t1_cols = ['Level', 'Split', 'N Total', 'N Need Correction', 'Pct Need Correction']
assert table1_export[req_t1_cols].notna().all().all(), 'Table 1 has nulls in required columns'

metric_cols = ['P', 'R', 'F1']
assert table2_export[metric_cols].notna().all().all(), 'Table 2 has missing metrics'

expected_table2_prefix = [
    ('train', 'day', 'na', 'm7_dtr'),
    ('train', 'day', 'na', 'm8_xgb'),
    ('train', 'interval', 'all_days', 'm7_dtr'),
]
actual_table2_prefix = list(
    table2_export[['Split', 'Model Level', 'Interval Scope', 'Method']].itertuples(index=False, name=None)
)[:3]
assert actual_table2_prefix == expected_table2_prefix, f'Unexpected Table 2 ordering prefix: {actual_table2_prefix}'

assert 'Year' not in table3_export.columns, 'Table 3 should not include year'
assert 'Year' not in table4_export.columns, 'Table 4 should not include year'
assert set(table3_export['Site Name']) == {'network', 'Site X', 'Site Y', 'Site Z'}, 'Unexpected Table 3 site names'
assert set(table4_export['Site Name']) == {'Site X', 'Site Y', 'Site Z'}, 'Unexpected Table 4 site names'
assert 'network' not in set(table4_export['Site Name']), 'Table 4 must exclude network row'

for tname, tdf in [
    ('Table 1', table1_export),
    ('Table 2', table2_export),
    ('Table 3', table3_export),
    ('Table 4', table4_export),
]:
    assert all('_' not in c for c in tdf.columns), f'{tname} headers should not contain underscore'
    assert all((len(c) == 0 or c[0].isupper()) for c in tdf.columns), f'{tname} headers should start with capital letter'

print('Validation passed.')
print('Table 1 rows:', len(table1_export))
print('Table 2 rows:', len(table2_export))
print('Table 3 rows:', len(table3_export))
print('Table 4 rows:', len(table4_export))

table1_export

Validation passed.
Table 1 rows: 4
Table 2 rows: 12
Table 3 rows: 4
Table 4 rows: 3


,Level,Split,N Total,N Need Correction,Pct Need Correction
0,day,train,6986,2095,29.989
1,day,test,3657,1328,36.314
2,interval,train,666432,27678,4.153
3,interval,test,344832,20302,5.888
